# Exploratory Data Analysis — Telco Customer Churn

**Phase 3.** This notebook is the readable narrative of the EDA. The analytical
logic lives in `src/churn/analysis/`; here we call it, look at the results and
interpret them.

Findings are labeled:

- **OBSERVATION** — measured directly in this sample.
- **HYPOTHESIS** — a plausible reading that still needs validation.
- **CANDIDATE** — something to test formally in a later phase.

Nothing here is causal evidence. The dataset is an observational snapshot, so
every relationship below is an association *within this sample*.

**What this notebook does not do:** no cleaning, no imputation decision, no
encoding, no scaling, no train/test split, no model. `TotalCharges` is coerced to
numeric in memory only — the raw file is never modified.

The generated report with every number and figure is `reports/eda_report.md`
(produced by `scripts/run_eda.py`).

In [ ]:
%matplotlib inline

import pandas as pd

from churn.analysis import descriptive as desc
from churn.analysis.association import (
    association_ranking,
    interpret_cramers_v,
    numeric_comparison,
)
from churn.analysis.frames import CHURN_FLAG, TENURE_BAND, TENURE_BAND_EDGES, load_eda_frame
from churn.analysis.plots import (
    plot_boxplot_by_group,
    plot_churn_rate_bars,
    plot_distribution_by_churn,
    plot_effect_sizes,
    plot_rate_heatmap,
    plot_scatter_by_churn,
    plot_target_distribution,
    use_project_style,
)

use_project_style()
pd.set_option("display.width", 140)

In [ ]:
frame = load_eda_frame()
baseline = desc.overall_churn_rate(frame)
print(f"{len(frame):,} rows x {frame.shape[1]} columns (2 are EDA helpers)")
print(f"population churn rate: {baseline:.4%}")
frame.head(3)

## 1. Target baseline

**Question:** how frequent is churn, and what does that imply for evaluation?

In [ ]:
counts = frame["Churn"].value_counts()
display(pd.DataFrame({"customers": counts, "share": counts / len(frame)}))
print(f"class ratio (retained per churner): {counts['No'] / counts['Yes']:.2f}")

plot_target_distribution(
    counts, "Churn is imbalanced: roughly one churner for every three customers"
)

**OBSERVATION.** The positive class is the minority. Predicting "no churn" for
everyone already scores about 73% accuracy while catching zero churners.

**DECISION (deferred).** This fixes the *evaluation frame*, not a technique:
accuracy is out as a selection metric, PR-AUC and recall/precision become the
relevant views, and the split must be stratified. Whether to use class weights or
resampling is a Phase 7 question.

## 2. Tenure

**Question:** does the length of the relationship relate to churn?

The tenure bands below follow the contract cycles the product offers rather than
equal widths. They are a **reading aid only** and carry no modeling commitment.

In [ ]:
display(desc.numeric_summary_by_target(frame, "tenure"))
print(numeric_comparison(frame, "tenure"))
print(f"band edges (months): {TENURE_BAND_EDGES}")

tenure_rates = desc.churn_rate_by(frame, TENURE_BAND, sort=False)
display(tenure_rates)

plot_distribution_by_churn(
    frame,
    "tenure",
    CHURN_FLAG,
    "Churned customers concentrate in the first months of the relationship",
    "Tenure (months)",
)

In [ ]:
plot_churn_rate_bars(
    tenure_rates,
    "Churn rate falls monotonically as tenure grows",
    "Tenure band (months, descriptive cuts)",
    baseline,
)

**OBSERVATION.** Churn rate declines monotonically across all five bands, and the
rank-biserial effect size is the largest of any numeric variable here.

**HYPOTHESIS.** Two mechanisms produce the same curve and this data cannot
separate them: risk genuinely decaying as the relationship matures, or the early
months filtering out a segment that was never going to stay (survivorship).

## 3. Contract

**Questions:** is contract flexibility associated with churn? Does the pattern
survive when tenure is taken into account? Are new customers concentrated in a
particular contract type?

In [ ]:
display(desc.churn_rate_by(frame, "Contract"))
display(frame.groupby("Contract", observed=True)["tenure"].describe()[["25%", "50%", "75%"]])

plot_churn_rate_bars(
    desc.churn_rate_by(frame, "Contract"),
    "Contract type shows the strongest association with churn",
    "Contract",
    baseline,
)

In [ ]:
rates, sizes = desc.churn_rate_matrix(frame, "Contract", TENURE_BAND)
display(rates.style.format("{:.1%}"))
display(sizes)

plot_rate_heatmap(
    rates,
    sizes,
    "The contract gap persists inside every tenure band",
    "Tenure band (months)",
    "Contract",
)

**OBSERVATION.** The gap survives conditioning on tenure: month-to-month customers
churn far more than two-year customers *inside every band*. Contract is therefore
not merely a proxy for tenure.

**OBSERVATION.** New customers are heavily concentrated in month-to-month. The
0% cells for two-year contracts in the early bands rest on a few dozen customers
— read them with the size table, not as evidence of a zero rate.

**HYPOTHESIS.** Commitment length and churn propensity are plausibly
co-determined: a long contract both restricts leaving and is chosen by customers
who already intend to stay. This sample cannot separate the contractual barrier
from the self-selection.

## 4. Services

**Question:** which subscribed services separate churners from those who stay?

`No internet service` and `No phone service` are **product states, not missing
values**. Every `X = No internet service` group is the same 1,526 customers, so
the consolidated chart shows that state once instead of seven identical bars.

In [ ]:
SERVICES = [
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]
# Every `X = No internet service` group is the same 1,526 customers, already shown
# as `InternetService = No`; the same holds for `No phone service`.
SENTINELS = ("No internet service", "No phone service")

rows = []
for column in SERVICES:
    table = desc.churn_rate_by(frame, column, sort=False)
    for category, row in table.iterrows():
        if str(category) in SENTINELS:
            continue
        rows.append(
            {
                "label": f"{column} = {category}",
                "n": int(row["n"]),
                "churn_rate": row["churn_rate"],
            }
        )

service_rates = pd.DataFrame(rows).set_index("label").sort_values("churn_rate", ascending=False)
display(service_rates.head(10))

plot_churn_rate_bars(
    service_rates,
    "Churn rate by service option (sentinel states shown once)",
    "Service option",
    baseline,
    horizontal=True,
)

**OBSERVATION.** Fiber optic subscribers churn far above DSL. Customers with no
internet at all show the *lowest* rate in the dataset — a sentinel category that
carries real information and must not be treated as `NaN`.

**OBSERVATION.** The four protective services (`OnlineSecurity`, `OnlineBackup`,
`DeviceProtection`, `TechSupport`) all show markedly lower churn among holders,
while both streaming services sit close to the baseline.

**HYPOTHESIS.** Protective services may act as switching friction, or may simply
mark more engaged customers. Streaming add-ons look like neutral entertainment
purchases with no retention signal.

## 5. Billing and payment

**Question:** do churners pay more — and if the totals say so, does it survive
conditioning on what they actually bought?

In [ ]:
for column in ("MonthlyCharges", "TotalCharges"):
    display(desc.numeric_summary_by_target(frame, column))
    print(numeric_comparison(frame, column))

plot_distribution_by_churn(
    frame,
    "MonthlyCharges",
    CHURN_FLAG,
    "Churned customers are concentrated in the higher monthly-charge range",
    "Monthly charges (US$)",
)

In [ ]:
display(desc.churn_rate_by(frame, "PaymentMethod"))
display(desc.churn_rate_by(frame, "PaperlessBilling"))

rates, sizes = desc.churn_rate_matrix(frame, "PaymentMethod", "Contract")
plot_rate_heatmap(
    rates,
    sizes,
    "Electronic check keeps a higher rate within every contract type",
    "Contract",
    "Payment method",
)

In [ ]:
display(
    frame.groupby(["InternetService", "Churn"], observed=True)["MonthlyCharges"].agg(
        ["size", "median"]
    )
)

plot_boxplot_by_group(
    frame,
    "MonthlyCharges",
    "InternetService",
    CHURN_FLAG,
    "Within each internet tier, churners do not pay more than those who stay",
    "Internet service",
    "Monthly charges (US$)",
)

**OBSERVATION.** Churners pay more per month but have accumulated less in total —
the two point in opposite directions because `TotalCharges` is dominated by
tenure.

**OBSERVATION — the marginal association reverses under conditioning.** Split by
internet tier, the median monthly charge of churners is *not* above that of those
who stay. The direction of the `MonthlyCharges`–churn association at population
level is not preserved once the tier is held fixed.

**What follows, and what does not.** The data do **not** support a simple reading
in which a higher `MonthlyCharges` on its own accounts for higher churn.
Confounding by service type is a plausible explanation — churners are
over-represented in the expensive fiber tier — but this analysis cannot establish
that it *is* the explanation, only that the marginal comparison is not
interpretable on its own.

**OBSERVATION.** Electronic check churns far above the automatic methods, and the
gap persists inside every contract type.

**HYPOTHESIS.** Price sensitivity remains possible and is not ruled out here.
Separating it from tier composition would require controlling for the other
determinants of tier choice, or a design capable of supporting causal claims —
neither of which this observational snapshot provides. Manual payment and the
fiber tier may also proxy for a lower-commitment customer profile; that too is an
untested reading.

## 6. TotalCharges investigation

**Question:** what exactly are the 11 unreadable cells, and how do the three
charge-related quantities relate?

Do **not** assume `TotalCharges = MonthlyCharges * tenure`: the monthly charge is
the *current* price while the total is *accumulated history*.

In [ ]:
blanks = frame[frame["TotalCharges"].isna()]
print(f"blank TotalCharges: {len(blanks)}")
print(f"of which tenure == 0: {(blanks['tenure'] == 0).sum()}")
print(f"rows with tenure == 0 in the whole file: {(frame['tenure'] == 0).sum()}")
display(
    blanks[["tenure", "MonthlyCharges", "Contract", "PhoneService", "InternetService", "Churn"]]
)

In [ ]:
complete = frame[frame["TotalCharges"].notna() & (frame["tenure"] > 0)]
implied = complete["tenure"] * complete["MonthlyCharges"]
ratio = complete["TotalCharges"] / implied

by_tenure = complete["TotalCharges"].corr(complete["tenure"], method="spearman")
by_implied = complete["TotalCharges"].corr(implied, method="spearman")
print(f"Spearman TotalCharges ~ tenure:                  {by_tenure:.4f}")
print(f"Spearman TotalCharges ~ tenure * MonthlyCharges: {by_implied:.4f}")
display(ratio.describe())

plot_scatter_by_churn(
    frame,
    "tenure",
    "TotalCharges",
    CHURN_FLAG,
    "Total charges grow with tenure; the 11 blank records sit exactly at tenure 0",
    "Tenure (months)",
    "Total charges (US$)",
    highlight_x=blanks["tenure"],
    highlight_y=pd.Series([0.0] * len(blanks), index=blanks.index),
    highlight_label="Blank TotalCharges (drawn at 0)",
)

**OBSERVATION.** The 11 blank records are exactly the 11 customers with
`tenure == 0` — the sets coincide. All are still active and all carry a non-zero
monthly charge.

**OBSERVATION.** `TotalCharges` tracks `tenure * MonthlyCharges` very closely but
is not equal to it: the ratio spreads roughly 0.69–1.57. That is what accumulated
history looks like when the monthly price changed over the relationship.

**HYPOTHESIS.** A blank marks a customer who has not been billed yet, rather than
a recording error.

**CANDIDATE treatments for Phase 4** — none chosen here: set to `0`; drop the 11
rows; statistical imputation (semantically wrong: it invents billing history);
or drop the column as near-redundant. Whatever is chosen must be fitted inside
the pipeline, after the split.

## 7. Customer profile

**Question:** do demographics separate churners — and where do they *not*?

In [ ]:
for column in ("gender", "SeniorCitizen", "Partner", "Dependents"):
    display(desc.churn_rate_by(frame, column))

**OBSERVATION.** `gender` shows essentially no association (Cramér's V ≈ 0.008,
p ≈ 0.49). The absence of a pattern is a result, not a hole in the analysis.

**OBSERVATION.** Senior citizens churn visibly more, but they are only ~16% of the
population. Customers without a partner or without dependents also churn more,
with weak effect sizes.

## 8. Selected interaction: TechSupport × InternetService

**Prior:** tech support looked protective, but fiber customers both churn more
*and* buy support less often. Does the support effect survive inside each tier?

In [ ]:
rates, sizes = desc.churn_rate_matrix(frame, "InternetService", "TechSupport")
display(rates.style.format("{:.1%}", na_rep="—"))

plot_rate_heatmap(
    rates,
    sizes,
    "Tech support is associated with lower churn inside both internet tiers",
    "Tech support",
    "Internet service",
)

**OBSERVATION.** The association holds inside both DSL and fiber, so it is not a
fiber artefact.

## 9. Statistical evidence — effect size, not p-value

With 7,043 observations nearly everything reaches significance. The ranking that
matters is by **magnitude**.

In [ ]:
CATEGORICALS = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    *SERVICES,
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]
ranking = association_ranking(frame, CATEGORICALS, "Churn")
ranking["effect"] = ranking["cramers_v"].map(interpret_cramers_v)
display(ranking)

plot_effect_sizes(ranking, "Categorical features ranked by effect size, not by p-value")

**OBSERVATION — significance is not magnitude.** `MultipleLines` reaches
p ≈ 0.003, "significant" at any conventional level, with an effect size of ≈0.04
— negligible. Ranking features by p-value would have promoted it for nothing.
That is the concrete reason p-values are not used as a feature ranking here.

**Multiple testing.** Sixteen categorical associations and three numeric
comparisons were tested against the same target on the same sample, and **no
formal multiple-testing correction (Bonferroni, Holm, Benjamini-Hochberg or
otherwise) was applied**. The p-values above are therefore *exploratory*: they
flag where a difference is unlikely to be sampling noise, nothing more. They are
never used in isolation to rank or select features — effect size and practical
relevance take precedence. Any inferential claim needing calibrated error rates
would have to be re-tested with an explicit correction, on data not used to
generate the hypothesis.

## 10. Feature-engineering candidates (conceptual only)

Nothing below is implemented. The point is to check whether an idea has any
empirical support *before* Phase 6 spends effort on it.

In [ ]:
internet = frame[frame["InternetService"] != "No"].copy()
internet["protective_services"] = desc.count_yes(
    internet, ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
)
print("Protective services (internet subscribers only):")
display(desc.churn_rate_by(internet, "protective_services", sort=False))

naive = frame.copy()
naive["service_count"] = (
    desc.count_yes(naive, [c for c in SERVICES if c not in ("PhoneService", "InternetService")])
    + (naive["PhoneService"] == "Yes").astype(int)
    + (naive["InternetService"] != "No").astype(int)
)
print("Naive total service count (whole population):")
display(desc.churn_rate_by(naive, "service_count", sort=False))

In [ ]:
plot_churn_rate_bars(
    desc.churn_rate_by(internet, "protective_services", sort=False),
    "Among internet subscribers, churn falls as protective services accumulate",
    "Number of protective services held (of 4)",
    baseline,
)

**CANDIDATE — count of protective services: investigate.** Among internet
subscribers the churn rate falls monotonically from ~57% at zero to ~5% at four.
No leakage risk: all four columns are known at signup. Redundancy with the source
columns is high and must be tested, not assumed.

**CANDIDATE — naive total service count: discard as a plain count.** The rate is
*non-monotonic* (it rises, peaks around three services, then falls) and Spearman
against churn is ≈0. Volume is not the signal; composition is.

**CANDIDATE — tenure × contract, charge intensity relative to tier median,
automatic-vs-manual payment flag: investigate.** Each has an explicit hypothesis
from the sections above. `TotalCharges / tenure` is worth a look but is undefined
at `tenure == 0` and must be handled inside the pipeline.

## Limitations and what comes next

- **Observational data.** Every relationship here is association, never causation.
- **Survivorship.** Long-tenure customers are by construction those who did not
  leave; rates conditioned on high tenure are not "risk after four years".
- **`MonthlyCharges` is the current price**, compared against an accumulated past.
- **No cost data.** No retention cost, customer value or campaign capacity exists
  here, so no threshold or business-impact claim can be made.

### No temporal dimension

The file carries no observation timestamp, no churn date and no explicit
prediction window. `tenure` is a duration recorded at snapshot time, not a
history. Therefore:

- prospective temporal validation is **not implementable** — there is no time
  ordering to split on;
- the task is a **snapshot classification approximation** of churn, not
  forecasting over a defined horizon;
- **no claim may be made that this protocol reproduces a real churn-prediction
  system**, which would predict churn within a stated window from features
  observed before it;
- no horizon is invented to compensate — the dataset does not contain one.

This limitation stays documented in the academic and portfolio deliverables.

### Analyst exposure to the full sample

This EDA ran on all 7,043 rows, the future holdout included. Precisely:

- **no fitting, preprocessing, model selection or tuning was performed** — nothing
  was learned from the file in a form a model could inherit;
- **however, the hypotheses and candidate features above were informed by the
  complete dataset**, holdout rows included;
- from Phase 4 onward the test set is protected against *fitting and tuning*, and
  **holdout metrics are not consulted until the final evaluation in Phase 9**;
- it should not be described as **completely unseen** in the analyst-exposure
  sense: protected from fitting, yes; never looked at, no;
- this is a **limitation of the protocol adopted**, not an oversight. Splitting
  before any exploration and confining the EDA to the training partition was a
  defensible alternative that was not chosen.

Because the complete dataset informed the exploratory hypotheses and candidate
features, the future holdout estimate **may be subject to optimistic bias from
analyst exposure**. Whether such bias actually materialises, and how large it
would be, **cannot be quantified in this experiment** — there is no independent
sample against which to measure it. It is recorded as a methodological limitation,
not as a correction factor. Regardless, candidate features must earn their place
through cross-validation on the training set, not through the numbers in this
notebook.

### Multiple testing

Sixteen categorical associations and three numeric comparisons were tested on the
same sample with **no formal correction applied**. The p-values are exploratory
evidence, not calibrated inferential statements.

The full generated write-up, with every figure and number, is
`reports/eda_report.md`.